In [0]:
%sql
USE CATALOG medalhao;

In [0]:
from pyspark.sql import functions as F
import requests
import json
from datetime import datetime

In [0]:
# Definição do caminho do schema Bronze
catalogo = "medalhao"
bronze_db_name = "bronze"

In [0]:
%sql
CREATE DATABASE IF NOT EXISTS bronze;
CREATE DATABASE IF NOT EXISTS silver;

In [0]:
spark.sql("DROP TABLE IF EXISTS bronze.ft_consumidores")
spark.sql("DROP TABLE IF EXISTS bronze.ft_geolocalizacao")
spark.sql("DROP TABLE IF EXISTS bronze.ft_itens_pedidos")
spark.sql("DROP TABLE IF EXISTS bronze.ft_pagamentos_pedidos")
spark.sql("DROP TABLE IF EXISTS bronze.ft_avaliacoes_pedidos")
spark.sql("DROP TABLE IF EXISTS bronze.ft_pedidos")
spark.sql("DROP TABLE IF EXISTS bronze.ft_produtos")
spark.sql("DROP TABLE IF EXISTS bronze.ft_vendedores")
spark.sql("DROP TABLE IF EXISTS bronze.dm_categoria_produtos_traducao")
spark.sql("DROP TABLE IF EXISTS bronze.dm_cotacao_dolar")



In [0]:

volume_path = "/Volumes/medalhao/default/landing"
arquivos_tabelas = [
    ("olist_customers_dataset.csv", "ft_consumidores"),
    ("olist_geolocation_dataset.csv", "ft_geolocalizacao"),
    ("olist_order_items_dataset.csv", "ft_itens_pedidos"),
    ("olist_order_payments_dataset.csv", "ft_pagamentos_pedidos"),
    ("olist_order_reviews_dataset.csv", "ft_avaliacoes_pedidos"),
    ("olist_orders_dataset.csv", "ft_pedidos"),
    ("olist_products_dataset.csv", "ft_produtos"),
    ("olist_sellers_dataset.csv", "ft_vendedores"),
    ("product_category_name_translation.csv", "dm_categoria_produtos_traducao")
]

In [0]:
def ingest_csv_to_bronze(csv_file, table_name):
    try:
        csv_path = f"{volume_path}/{csv_file}"
        
        df = spark.read.csv(csv_path, header=True, inferSchema=True)

        if df.count() == 0:
            raise ValueError(f"O arquivo {csv_file} está vazio ou não pôde ser lido.")
        
        df_with_timestamp = df.withColumn("ingestion_timestamp", F.current_timestamp())
        
        df_with_timestamp.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{bronze_db_name}.{table_name}")
                           
        print(f"✅ Tabela bronze.{table_name} criada com sucesso!\n")
        
    except Exception as e:
        print(f"Erro ao processar '{csv_file}' para '{table_name}': {e}")

In [0]:
print("Iniciando ingestão das 9 tabelas CSV para a Camada Bronze...")

for csv_file, table_name in arquivos_tabelas:
    ingest_csv_to_bronze(csv_file, table_name)

print("\nIngestão dos arquivos CSV concluída!")

In [0]:


data_inicio = "01-01-2016"
data_fim = "10-10-2019"

# Construção da URL (tudo em uma única chamada)
url = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
    f"?@dataInicial='{data_inicio}'"
    f"&@dataFinalCotacao='{data_fim}'"
    "&$top=10000"  # aumenta o limite padrão de 100 registros
    "&$format=json"
)

response = requests.get(url)
data = response.json()

# Converte para DataFrame Spark
df_cotacao = spark.createDataFrame(data["value"])
display(df_cotacao.limit(5))

In [0]:

from pyspark.sql import functions as F

# Adicionar coluna de ingestão
df_cotacao = (
    df_cotacao
    .withColumn("ingestion_timestamp", F.current_timestamp())
)

# Salvar tabela das cotações no formato Delta
df_cotacao.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{bronze_db_name}.dm_cotacao_dolar")

print("Tabela bronze.dm_cotacao_dolar criada com sucesso!")

In [0]:
import os
import platform
hostname = "google.com"
param = "-n 4" if platform.system().lower() == "windows" else "-c 4"
command = f"ping {param} {hostname}"
print(f"Executando comando: {command}")
print("-" * 30)
os.system(command)